In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px  # interactive figures
import plotly.graph_objects as go
import seaborn as sns


In [2]:
import pandas as pd

df = pd.read_csv("../data_raw/date_24_art_loreal_20%.csv")

df = df[df["ID_ARTICOL"] == 94869]

df.head()

,DATA,ID_ARTICOL,ARTICOL,CANTITATE,VAL_IESIRE_FARA_TVA,VAL_IESIRE_CU_TVA,VAL_INTRARE_FARA_TVA,TOTAL_DISCOUNT,RATA_DISCOUNT,ADAOS,STOC_INITIAL,STOC_FINAL,RUPTURA_STOC
4,01-Jan-24,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,0,0.00,0.0,0.00,0.0,0.0,0.00,168,168,0
14,02-Jan-24,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,0,0.00,0.0,0.00,0.0,0.0,0.00,168,168,0
38,03-Jan-24,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,10,571.46,680.0,508.30,0.0,0.0,63.16,168,158,0
49,04-Jan-24,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,4,228.57,272.0,203.32,0.0,0.0,25.25,158,214,0
52,05-Jan-24,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,12,685.69,816.0,609.96,0.0,0.0,75.73,214,202,0


standard deviation is a way to measure how "spread out" or "different" the values in a group are from the average.

If the standard deviation is small, it means most values are close to the average (not much variation).
If the standard deviation is large, it means the values are spread out and vary a lot from the average.

Conversie coloana data din object in datetime

In [3]:
df['DATA'] = pd.to_datetime(df['DATA'], errors="coerce")

C:\Users\Mara\AppData\Local\Temp\ipykernel_41108\2340964975.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['DATA'] = pd.to_datetime(df['DATA'], errors="coerce")


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 733 entries, 4 to 9992
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   DATA                  733 non-null    datetime64[ns]
 1   ID_ARTICOL            733 non-null    int64         
 2   ARTICOL               733 non-null    object        
 3   CANTITATE             733 non-null    int64         
 4   VAL_IESIRE_FARA_TVA   733 non-null    float64       
 5   VAL_IESIRE_CU_TVA     733 non-null    float64       
 6   VAL_INTRARE_FARA_TVA  733 non-null    float64       
 7   TOTAL_DISCOUNT        733 non-null    float64       
 8   RATA_DISCOUNT         706 non-null    float64       
 9   ADAOS                 733 non-null    float64       
 10  STOC_INITIAL          733 non-null    int64         
 11  STOC_FINAL            733 non-null    int64         
 12  RUPTURA_STOC          733 non-null    int64         
dtypes: datetime64[ns](1), fl

In [5]:
# Sortare logică pentru time series
df = df.sort_values(by=["DATA"]).reset_index(drop=True)

In [6]:
# reguli de calitate
checks = {}

# 1) vânzare > 0 dar valoare = 0 (suspect)
checks["qty_pos_val_zero"] = df[(df["CANTITATE"] > 0) & (df["VAL_IESIRE_CU_TVA"] == 0)]

# 2) valoare > 0 dar cantitate = 0 (suspect)
checks["val_pos_qty_zero"] = df[(df["VAL_IESIRE_CU_TVA"] > 0) & (df["CANTITATE"] == 0)]

# 3) stoc final negativ (dacă există în dataset)
if "STOC_FINAL" in df.columns:
    checks["stoc_final_negativ"] = df[df["STOC_FINAL"] < 0]

# 4) discount > 0 dar rata_discount = 0 (suspect)
if "TOTAL_DISCOUNT" in df.columns and "RATA_DISCOUNT" in df.columns:
    checks["disc_pos_rata_zero"] = df[(df["TOTAL_DISCOUNT"] > 0) & (df["RATA_DISCOUNT"] == 0)]

# 5) 
checks["stockout_start"] = df[(df["STOC_INITIAL"] == 0) & (df["CANTITATE"] == 0)]
checks["no_sales_but_in_stock"] = df[(df["STOC_INITIAL"] > 0) & (df["CANTITATE"] == 0)]

{k: v.shape[0] for k, v in checks.items()}

{'qty_pos_val_zero': 1,
 'val_pos_qty_zero': 0,
 'stoc_final_negativ': 0,
 'disc_pos_rata_zero': 1,
 'stockout_start': 1,
 'no_sales_but_in_stock': 24}

In [7]:
cols = ["CANTITATE", "STOC_INITIAL", "STOC_FINAL", "VAL_IESIRE_FARA_TVA"]

(df[cols] < 0).sum()


CANTITATE              0
STOC_INITIAL           0
STOC_FINAL             0
VAL_IESIRE_FARA_TVA    0
dtype: int64

In [8]:
# df = df.set_index('DATA') 

In [9]:
# df

Pentru evitarea redundanței informaționale, variabilele puternic corelate vor fi filtrate.  ( gen val iesire cu fara tva, total discount - rata discount )

In [10]:
df_feat = df.drop(columns=[ "STOC_FINAL", "RUPTURA_STOC", "VAL_IESIRE_CU_TVA", "RATA_DISCOUNT"])

df_feat


,DATA,ID_ARTICOL,ARTICOL,CANTITATE,VAL_IESIRE_FARA_TVA,VAL_INTRARE_FARA_TVA,TOTAL_DISCOUNT,ADAOS,STOC_INITIAL
0,2024-01-01,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,0,0.00,0.00,0.0,0.00,168
1,2024-01-02,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,0,0.00,0.00,0.0,0.00,168
2,2024-01-03,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,10,571.46,508.30,0.0,63.16,168
3,2024-01-04,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,4,228.57,203.32,0.0,25.25,158
4,2024-01-05,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,12,685.69,609.96,0.0,75.73,214
...,...,...,...,...,...,...,...,...,...
728,2025-12-31,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,1,61.99,54.15,0.0,7.84,243
729,2026-01-01,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,0,0.00,0.00,0.0,0.00,242
730,2026-01-02,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,0,0.00,0.00,0.0,0.00,242
731,2026-01-03,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,6,371.88,324.90,0.0,46.98,242


In [11]:

df_feat["DOW"] = df_feat["DATA"].dt.dayofweek          # 0..6
df_feat["ESTE_WEEKEND"] = (df_feat["DOW"] >= 5).astype(int)

# dacă ai deja LUNA ca yyyy-mm, păstreaz-o ca perioadă (opțional)
# df_feat["YYYY_MM"] = df_feat["DATA"].dt.to_period("M").astype(str)

# opțional: zi din lună (uneori surprinde patternuri de început/sfârșit)
df_feat["ZI_DIN_LUNA"] = df_feat["DATA"].dt.day

df_feat


,DATA,ID_ARTICOL,ARTICOL,CANTITATE,VAL_IESIRE_FARA_TVA,VAL_INTRARE_FARA_TVA,TOTAL_DISCOUNT,ADAOS,STOC_INITIAL,DOW,ESTE_WEEKEND,ZI_DIN_LUNA
0,2024-01-01,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,0,0.00,0.00,0.0,0.00,168,0,0,1
1,2024-01-02,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,0,0.00,0.00,0.0,0.00,168,1,0,2
2,2024-01-03,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,10,571.46,508.30,0.0,63.16,168,2,0,3
3,2024-01-04,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,4,228.57,203.32,0.0,25.25,158,3,0,4
4,2024-01-05,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,12,685.69,609.96,0.0,75.73,214,4,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...
728,2025-12-31,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,1,61.99,54.15,0.0,7.84,243,2,0,31
729,2026-01-01,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,0,0.00,0.00,0.0,0.00,242,3,0,1
730,2026-01-02,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,0,0.00,0.00,0.0,0.00,242,4,0,2
731,2026-01-03,94869,CERAVE LOTIUNE HIDRATANTA PENTRU FATA SI CORP ...,6,371.88,324.90,0.0,46.98,242,5,1,3


In [12]:
fig = px.line(df, x='DATA', y='CANTITATE', title='Time Series with Range Slider and Selectors')

fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
        buttons=list([
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(count=6, label="6m", step="month", stepmode="backward"),
            # dict(count=1, label="YTD", step="year", stepmode="todate"),
            dict(count=1, label="1y", step="year", stepmode="backward"),
            dict(step="all")
        ])
    )
)

fig.update_layout(width=1100, height=500)
fig.show()

## ARIMA

In [13]:
# df_art = df.set_index("DATA")["CANTITATE"].asfreq("D")
# ARIMA works with series, not dataframes
df_art = (
    df
    .set_index("DATA")["CANTITATE"]
    .astype(float)
    .sort_index()
)


In [14]:
df_art

DATA
2024-01-01     0.0
2024-01-02     0.0
2024-01-03    10.0
2024-01-04     4.0
2024-01-05    12.0
              ... 
2025-12-31     1.0
2026-01-01     0.0
2026-01-02     0.0
2026-01-03     6.0
2026-01-04     6.0
Name: CANTITATE, Length: 733, dtype: float64

In [15]:
df_art.map(type).value_counts().head(10)


CANTITATE
<class 'float'>    733
Name: count, dtype: int64

In [16]:
type(df_art)
df_art.name
df_art.index.name
df_art.dtype
df_art.head()


DATA
2024-01-01     0.0
2024-01-02     0.0
2024-01-03    10.0
2024-01-04     4.0
2024-01-05    12.0
Name: CANTITATE, dtype: float64

In [17]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

C:\Users\Mara\AppData\Roaming\Python\Python313\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning:

In the future `np.object` will be defined as the corresponding NumPy scalar.



In [18]:
# pip install tensorflow

In [29]:
len(df_art) - 30

703

In [30]:
forecast_horizon = 30
n = len(df_art)
if forecast_horizon >= n:
    raise ValueError("forecast_horizon must be smaller than the series length")

split_idx = n - forecast_horizon

split_idx


703

In [31]:
train = df_art.iloc[:split_idx]
test  = df_art.iloc[split_idx:]

In [34]:
# import libraries
from statsmodels.tsa.stattools import adfuller

# ADF test
result = adfuller(train.dropna())
print('ADF Statistic:', result[0])
print('p-value:', result[1])

# if p-value > 0.05, the series is non-stationary and needs differencing
if result[1] > 0.05:
    print("needs differencing")
else:
    print("time series is stationary")

ADF Statistic: -4.315255411453733
p-value: 0.0004179537924403509
time series is stationary


In [35]:
from pmdarima import auto_arima

model_auto = auto_arima(
    train,
    seasonal=False,
    d=None,
    stepwise=True,
    suppress_warnings=True,
    trace=True
)

selected_order = model_auto.order
selected_order


Performing stepwise search to minimize aic
 ARIMA(2,1,2)(0,0,0)[0] intercept   : AIC=3731.775, Time=0.35 sec
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=3969.040, Time=0.02 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=3875.619, Time=0.07 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=3744.169, Time=0.07 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=3967.041, Time=0.02 sec
 ARIMA(1,1,2)(0,0,0)[0] intercept   : AIC=3730.547, Time=0.22 sec
 ARIMA(0,1,2)(0,0,0)[0] intercept   : AIC=3732.446, Time=0.15 sec
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=3735.650, Time=0.12 sec
 ARIMA(1,1,3)(0,0,0)[0] intercept   : AIC=3731.468, Time=0.33 sec
 ARIMA(0,1,3)(0,0,0)[0] intercept   : AIC=3729.643, Time=0.13 sec
 ARIMA(0,1,4)(0,0,0)[0] intercept   : AIC=3731.558, Time=0.15 sec
 ARIMA(1,1,4)(0,0,0)[0] intercept   : AIC=3733.395, Time=0.41 sec
 ARIMA(0,1,3)(0,0,0)[0]             : AIC=3727.648, Time=0.07 sec
 ARIMA(0,1,2)(0,0,0)[0]             : AIC=3730.454, Time=0.06 sec
 ARIMA(1,1,3)(0,0,0)[0]          

(0, 1, 3)

In [36]:
from statsmodels.tsa.arima.model import ARIMA

arima = ARIMA(train, order=selected_order)
arima_fit = arima.fit()


C:\Users\Mara\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

C:\Users\Mara\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

C:\Users\Mara\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.



In [37]:
pred_arima = arima_fit.forecast(steps=forecast_horizon)


C:\Users\Mara\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning:

No supported index is available. Prediction results will be given with an integer index beginning at `start`.

C:\Users\Mara\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning:

No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.



In [38]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(test, pred_arima)

mse = mean_squared_error(test, pred_arima)
rmse = np.sqrt(mse)

r2 = r2_score(test, pred_arima)


mae = float(mae)
rmse = float(rmse)


mae, rmse, r2



(2.0368424175100315, 2.538641395322912, -0.01031703895529601)

In [26]:
import numpy as np

r = np.corrcoef(test, pred_arima)[0, 1]
float(r)
# arata daca modelul urmareste trandul, nu cat de exact prezice

-0.048371002849509105

In [27]:
import pandas as pd
import plotly.express as px

df_plot_arima = pd.DataFrame({
    "DATA": test.index,
    "Actual": test.values,
    "Predicted": pred_arima
})

df_long_arima = df_plot_arima.melt(
    id_vars="DATA",
    value_vars=["Actual", "Predicted"],
    var_name="Serie",
    value_name="Cantitate"
)

fig = px.line(
    df_long_arima,
    x="DATA",
    y="Cantitate",
    color="Serie",
    title="ARIMA – Actual vs Predicted"
)

fig.update_layout(
    xaxis_title="Data",
    yaxis_title="Cantitate vândută",
    hovermode="x unified",
    legend_title_text=""
)

fig.update_xaxes(rangeslider_visible=True)

fig.show()

